# 从零实现 GIN 图分类：可学习 $\epsilon$、多图 Batch 与图级读出

本 Notebook 只使用 PyTorch 基础张量与 `nn.Module`，手写 `GINLayer`、`GINClassifier`、无 PyG 的多图拼接以及图级 sum pooling；不使用 `torch_geometric`、DGL 或任何现成 GNN 层。

学习目标不是背一段网络定义，而是打通一条可审计链路：图合同 → 邻居求和 oracle → `(1+ε)` 中心节点更新 → 多图 batch 隔离 → 节点置换不变性 → 受控图分类训练 → 梯度与制品合同。所有样本均为离线合成小图，固定随机种子、CPU、单线程；结果只验证实现与评估协议，不代表真实业务泛化能力。

In [ ]:
from __future__ import annotations

import warnings
warnings.filterwarnings("ignore", message=".*pynvml.*", category=FutureWarning)

from dataclasses import dataclass
import hashlib
import json
import math
import random
import time

import numpy as np
import torch
from torch import nn
import torch.nn.functional as F

SEED = 3701
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.set_num_threads(1)
DEVICE = torch.device("cpu")
DTYPE = torch.float32

def canonical_hash(payload) -> str:
    raw = json.dumps(payload, ensure_ascii=False, sort_keys=True, separators=(",", ":"))
    return hashlib.sha256(raw.encode("utf-8")).hexdigest()[:20]

assert DEVICE.type == "cpu" and torch.get_num_threads() == 1
assert torch.initial_seed() == SEED
assert torch.__version__ and not any(name in globals() for name in ("torch_geometric", "dgl"))

## 1. 图样本与无泄漏切分

构造两类无向图：类别 0 是环，类别 1 是星形；节点数在 6–8 之间变化，节点初始特征只含常数项和很小的、与类别无关的局部标记。分类所需的主要信号因此来自拓扑，而不是把标签偷偷写进特征。

每类 18 张图：前 12 张训练、随后 3 张验证、最后 3 张测试。`graph_id` 在三个集合中互斥；validation 只选择 checkpoint，test 在模型冻结后只打开一次。这里的受控任务用于验证 GIN 能否学习，不能被包装成真实数据基准。

In [ ]:
@dataclass(frozen=True)
class GraphSample:
    graph_id: str
    x: torch.Tensor
    edge_pairs: tuple[tuple[int, int], ...]
    label: int
    split: str

def make_graph(label: int, index: int, split: str) -> GraphSample:
    if label not in (0, 1) or split not in {"train", "val", "test"}:
        raise ValueError("label/split 非法")
    n = 6 + index % 3
    if label == 0:
        pairs = {tuple(sorted((i, (i + 1) % n))) for i in range(n)}
    else:
        pairs = {(0, i) for i in range(1, n)}
    # 第二列仅是节点局部奇偶标记，不依赖图标签。
    x = torch.tensor([[1.0, 0.05 * (i % 2)] for i in range(n)], dtype=DTYPE)
    return GraphSample(f"{split}-c{label}-{index:02d}", x, tuple(sorted(pairs)), label, split)

graphs = []
for label in (0, 1):
    for i in range(18):
        split = "train" if i < 12 else ("val" if i < 15 else "test")
        graphs.append(make_graph(label, i, split))

split_graphs = {s: [g for g in graphs if g.split == s] for s in ("train", "val", "test")}
all_ids = [g.graph_id for g in graphs]
assert len(graphs) == 36 and len(all_ids) == len(set(all_ids))
assert tuple(len(split_graphs[s]) for s in ("train", "val", "test")) == (24, 6, 6)
assert all(set(g.label for g in split_graphs[s]) == {0, 1} for s in split_graphs)
assert not ({g.graph_id for g in split_graphs["train"]} & {g.graph_id for g in split_graphs["test"]})
assert all(g.x.shape[1] == 2 and torch.isfinite(g.x).all() for g in graphs)

## 2. 不依赖 PyG 的多图 Batch

稀疏多图 batch 不需要补齐节点：把各图节点沿第 0 维拼接，把第 $k$ 张图的边端点加上累计节点偏移，并维护 `batch[i]=k`。拼接后的邻接在数学上是块对角结构；任意一条边两端的 `batch` 必须相同，否则一张图会读取另一张图的消息。

对图级 sum pooling：$h_G=\sum_{v:\,batch(v)=G}h_v$。求和保留多重集合计数，是 GIN 表达力论证的重要组成；mean pooling 会抹去部分图规模信息。空图、越界边、重复/自环原始边、错 batch 都应 fail-closed，而不是依赖 `index_add_` 偶然报错。

In [ ]:
@dataclass(frozen=True)
class GraphBatch:
    x: torch.Tensor
    edge_index: torch.Tensor
    batch: torch.Tensor
    labels: torch.Tensor
    graph_ids: tuple[str, ...]
    num_graphs: int

def validate_sample(graph: GraphSample, feature_dim: int | None = None) -> None:
    if graph.x.ndim != 2 or graph.x.shape[0] == 0:
        raise ValueError("空图或 x 不是二维张量")
    if feature_dim is not None and graph.x.shape[1] != feature_dim:
        raise ValueError("多图特征维度不一致")
    if not torch.isfinite(graph.x).all():
        raise ValueError("节点特征包含非有限值")
    n = graph.x.shape[0]
    seen = set()
    for u, v in graph.edge_pairs:
        if not (0 <= u < n and 0 <= v < n) or u == v:
            raise ValueError("边端点越界或原始边含自环")
        key = tuple(sorted((u, v)))
        if key in seen:
            raise ValueError("无向边重复")
        seen.add(key)

def validate_batched_graph_tensors(x: torch.Tensor, edge_index: torch.Tensor,
                                   batch: torch.Tensor, num_graphs: int) -> None:
    if x.ndim != 2 or x.shape[0] == 0 or not torch.isfinite(x).all():
        raise ValueError("节点特征必须是非空有限二维张量")
    if edge_index.ndim != 2 or edge_index.shape[0] != 2 or edge_index.dtype != torch.long:
        raise ValueError("edge_index 必须是 shape=(2,E) 的 long 张量")
    if batch.ndim != 1 or batch.dtype != torch.long or batch.numel() != x.shape[0] or num_graphs <= 0:
        raise ValueError("batch/num_graphs 合同不匹配")
    if x.device != edge_index.device or x.device != batch.device:
        raise ValueError("x/edge_index/batch 必须位于同一设备")
    if edge_index.numel() and (int(edge_index.min()) < 0 or int(edge_index.max()) >= x.shape[0]):
        raise ValueError("edge_index 越界")
    if int(batch.min()) < 0 or int(batch.max()) >= num_graphs:
        raise ValueError("图编号越界")
    counts = torch.bincount(batch, minlength=num_graphs)
    if counts.numel() != num_graphs or bool((counts == 0).any()):
        raise ValueError("batch 声明了空图")
    if edge_index.numel() and not torch.equal(batch[edge_index[0]], batch[edge_index[1]]):
        raise ValueError("检测到跨图边污染")


def batch_graphs(items: list[GraphSample]) -> GraphBatch:
    if not items:
        raise ValueError("至少需要一张非空图")
    feature_dim = items[0].x.shape[1]
    xs, directed, owners, labels, ids = [], [], [], [], []
    offset = 0
    for gid, graph in enumerate(items):
        validate_sample(graph, feature_dim)
        xs.append(graph.x); labels.append(graph.label); ids.append(graph.graph_id)
        owners.append(torch.full((graph.x.shape[0],), gid, dtype=torch.long))
        for u, v in graph.edge_pairs:
            directed.extend([(offset + u, offset + v), (offset + v, offset + u)])
        offset += graph.x.shape[0]
    edge_index = (torch.tensor(directed, dtype=torch.long).T.contiguous()
                  if directed else torch.empty((2, 0), dtype=torch.long))
    out = GraphBatch(torch.cat(xs), edge_index, torch.cat(owners),
                     torch.tensor(labels, dtype=torch.long), tuple(ids), len(items))
    validate_batched_graph_tensors(out.x, out.edge_index, out.batch, out.num_graphs)
    return out

def sum_pool(x: torch.Tensor, batch: torch.Tensor, num_graphs: int) -> torch.Tensor:
    empty_edges = torch.empty((2, 0), dtype=torch.long, device=x.device)
    validate_batched_graph_tensors(x, empty_edges, batch, num_graphs)
    pooled = x.new_zeros((num_graphs, x.shape[1]))
    pooled.index_add_(0, batch, x)
    return pooled

probe_batch = batch_graphs([graphs[0], graphs[18]])
assert probe_batch.x.shape[0] == graphs[0].x.shape[0] + graphs[18].x.shape[0]
assert torch.equal(probe_batch.batch[probe_batch.edge_index[0]], probe_batch.batch[probe_batch.edge_index[1]])
assert torch.equal(sum_pool(torch.ones(probe_batch.x.shape[0], 1), probe_batch.batch, 2).squeeze(),
                   torch.tensor([float(graphs[0].x.shape[0]), float(graphs[18].x.shape[0])]))
try:
    batch_graphs([])
    raise AssertionError("空图列表未被拒绝")
except ValueError as exc:
    assert "至少需要" in str(exc)
try:
    sum_pool(torch.ones(2, 3), torch.tensor([0, 0]), 2)
    raise AssertionError("声明空图的 batch 未被拒绝")
except ValueError as exc:
    assert "空图" in str(exc)

## 3. 邻居求和：先固定消息方向

约定 `edge_index[0]=source`、`edge_index[1]=target`，因此

$$a_i=\sum_{j\in\mathcal N(i)}x_j.$$

下方 3 节点 oracle 只有无向边 `0—1—2`，输入为 `[1,2,4]`：节点 0 收到 2，节点 1 收到 $1+4=5$，节点 2 收到 2。这个断言能同时发现 source/target 颠倒、误加自环以及把 sum 写成 mean。

In [ ]:
def neighbor_sum(x: torch.Tensor, edge_index: torch.Tensor) -> torch.Tensor:
    if x.ndim != 2 or edge_index.ndim != 2 or edge_index.shape[0] != 2:
        raise ValueError("x 或 edge_index shape 非法")
    if edge_index.numel() and (int(edge_index.min()) < 0 or int(edge_index.max()) >= x.shape[0]):
        raise ValueError("edge_index 越界")
    source, target = edge_index
    out = torch.zeros_like(x)
    out.index_add_(0, target, x[source])
    return out

oracle_x = torch.tensor([[1.0], [2.0], [4.0]])
oracle_edges = torch.tensor([[0, 1, 1, 2], [1, 0, 2, 1]], dtype=torch.long)
oracle_agg = neighbor_sum(oracle_x, oracle_edges)
assert torch.equal(oracle_agg, torch.tensor([[2.0], [5.0], [2.0]]))
assert not torch.equal(oracle_agg, torch.tensor([[3.0], [7.0], [6.0]]))  # 没有偷偷加中心节点
assert torch.equal(neighbor_sum(torch.ones(2, 3), torch.empty((2, 0), dtype=torch.long)), torch.zeros(2, 3))

# 非对称单向链才真正能杀死 source/target 颠倒的实现：0→1、1→2。
direction_x = torch.tensor([[1.0], [10.0], [100.0]])
direction_edges = torch.tensor([[0, 1], [1, 2]], dtype=torch.long)
direction_expected = torch.tensor([[0.0], [1.0], [10.0]])
assert torch.equal(neighbor_sum(direction_x, direction_edges), direction_expected)
assert not torch.equal(neighbor_sum(direction_x, direction_edges.flip(0)), direction_expected)


## 4. `GINLayer`：可学习中心权重与 MLP

GIN 的一层为

$$h_i^{(k)}=\mathrm{MLP}^{(k)}\!\left((1+\epsilon^{(k)})h_i^{(k-1)}+\sum_{j\in\mathcal N(i)}h_j^{(k-1)}\right).$$

`eps` 可以固定为 0，也可以注册为标量参数；这里选择可学习版本。MLP 不是用一个线性层敷衍，而是显式实现 `Linear → ReLU → Linear → LayerNorm`。输入 `(N,F_in)`，输出 `(N,F_out)`；单层消息聚合时间复杂度约为 $O(EF_{in})$，MLP 为 $O(NF_{in}F_h)$，稀疏边存储为 $O(E)$。

In [ ]:
class GINMLP(nn.Module):
    def __init__(self, in_features: int, hidden_features: int, out_features: int):
        super().__init__()
        if min(in_features, hidden_features, out_features) <= 0:
            raise ValueError("MLP 维度必须为正")
        self.in_features = in_features
        self.fc1 = nn.Linear(in_features, hidden_features)
        self.fc2 = nn.Linear(hidden_features, out_features)
        self.norm = nn.LayerNorm(out_features)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        if x.ndim != 2 or x.shape[1] != self.in_features or not torch.isfinite(x).all():
            raise ValueError("MLP 输入 shape 或数值非法")
        return self.norm(self.fc2(F.relu(self.fc1(x))))

class GINLayer(nn.Module):
    def __init__(self, in_features: int, hidden_features: int, out_features: int,
                 initial_eps: float = 0.0, train_eps: bool = True):
        super().__init__()
        eps = torch.tensor(float(initial_eps), dtype=DTYPE)
        if train_eps:
            self.eps = nn.Parameter(eps)
        else:
            self.register_buffer("eps", eps)
        self.mlp = GINMLP(in_features, hidden_features, out_features)
        self.in_features = in_features

    def forward(self, x: torch.Tensor, edge_index: torch.Tensor) -> torch.Tensor:
        if x.ndim != 2 or x.shape[1] != self.in_features:
            raise ValueError("GINLayer 输入特征维度不匹配")
        aggregated = neighbor_sum(x, edge_index)
        return self.mlp((1.0 + self.eps) * x + aggregated)

gin_probe = GINLayer(1, 4, 3, initial_eps=0.25)
gin_out = gin_probe(oracle_x, oracle_edges)
assert gin_out.shape == (3, 3) and torch.isfinite(gin_out).all()
assert isinstance(gin_probe.eps, nn.Parameter) and gin_probe.eps.requires_grad
assert math.isclose(float(gin_probe.eps.detach()), 0.25, abs_tol=1e-7)
assert sum(p.numel() for p in gin_probe.parameters()) == 1 + (1*4+4) + (4*3+3) + 2*3

## 5. 图分类器：逐层表示与图级读出

三层 GIN 产生节点表示 $H^{(1)},H^{(2)},H^{(3)}$。每层都做 sum pooling，再把三个图向量拼接后分类，相当于保留不同感受野的信号。`forward` 接收拼接节点、块对角边和 batch 所属关系，输出 `(B,C)` logits；它不会在内部根据标签改变图结构。

GIN 对**节点编号置换**应保持图级输出不变，但这要求特征和边同时重编号。只打乱特征、不改边不是置换同一张图，而是在创建另一张有属性图。

In [ ]:
class GINClassifier(nn.Module):
    def __init__(self, in_features: int, hidden_features: int, num_classes: int, num_layers: int = 3):
        super().__init__()
        if num_layers < 1 or num_classes < 2:
            raise ValueError("层数或类别数非法")
        layers = []
        for layer_id in range(num_layers):
            in_dim = in_features if layer_id == 0 else hidden_features
            layers.append(GINLayer(in_dim, hidden_features, hidden_features))
        self.layers = nn.ModuleList(layers)
        self.classifier = nn.Linear(hidden_features * num_layers, num_classes)

    def forward(self, x: torch.Tensor, edge_index: torch.Tensor,
                batch: torch.Tensor, num_graphs: int) -> torch.Tensor:
        validate_batched_graph_tensors(x, edge_index, batch, num_graphs)
        pooled_layers = []
        h = x
        for layer in self.layers:
            h = F.relu(layer(h, edge_index))
            pooled_layers.append(sum_pool(h, batch, num_graphs))
        return self.classifier(torch.cat(pooled_layers, dim=-1))

torch.manual_seed(SEED + 1)
invariant_model = GINClassifier(2, 12, 2).eval()
sample = graphs[2]
base = batch_graphs([sample])
base_logits = invariant_model(base.x, base.edge_index, base.batch, 1)

perm = torch.tensor([2, 0, 5, 1, 4, 3, 6, 7][:sample.x.shape[0]])
old_to_new = torch.empty(sample.x.shape[0], dtype=torch.long)
old_to_new[perm] = torch.arange(sample.x.shape[0])
perm_pairs = tuple(sorted(tuple(sorted((int(old_to_new[u]), int(old_to_new[v])))) for u, v in sample.edge_pairs))
perm_sample = GraphSample("permuted", sample.x[perm], perm_pairs, sample.label, sample.split)
perm_batch = batch_graphs([perm_sample])
perm_logits = invariant_model(perm_batch.x, perm_batch.edge_index, perm_batch.batch, 1)
assert torch.allclose(base_logits, perm_logits, atol=2e-6)

pair = [graphs[1], graphs[20]]
together = batch_graphs(pair)
logits_together = invariant_model(together.x, together.edge_index, together.batch, 2)
logits_separate = torch.cat([
    invariant_model((one := batch_graphs([g])).x, one.edge_index, one.batch, 1) for g in pair
], dim=0)
assert torch.allclose(logits_together, logits_separate, atol=2e-6)
assert logits_together.shape == (2, 2)

# 模型边界也必须 fail closed，不能只信任 batch_graphs 的调用路径。
cross_source = int((together.batch == 0).nonzero(as_tuple=False)[0])
cross_target = int((together.batch == 1).nonzero(as_tuple=False)[0])
forged_edges = torch.cat([together.edge_index,
                          torch.tensor([[cross_source], [cross_target]], dtype=torch.long)], dim=1)
try:
    invariant_model(together.x, forged_edges, together.batch, 2)
    raise AssertionError("跨图边进入模型后未被拒绝")
except ValueError as exc:
    assert "跨图边" in str(exc)


## 6. 受控训练、checkpoint 与梯度

全部训练图组成一个稀疏 batch。每轮仅对 train 计算交叉熵；每 5 轮在 validation 上选择 checkpoint。为了让 Notebook 快速、确定地执行，模型很小且不使用 DataLoader 多进程。我们同时要求首层 `eps`、MLP 权重和分类头都收到有限非零梯度，避免“模型表面训练、消息层实际断路”。

In [ ]:
train_batch = batch_graphs(split_graphs["train"])
val_batch = batch_graphs(split_graphs["val"])
test_batch = batch_graphs(split_graphs["test"])

torch.manual_seed(SEED + 2)
model = GINClassifier(2, 20, 2, num_layers=3).to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=0.015, weight_decay=1e-5)

model.train()
initial_logits = model(train_batch.x, train_batch.edge_index, train_batch.batch, train_batch.num_graphs)
initial_loss = float(F.cross_entropy(initial_logits, train_batch.labels))
best_val, best_state = -1.0, None
train_start = time.perf_counter()
for epoch in range(141):
    model.train(); optimizer.zero_grad(set_to_none=True)
    logits = model(train_batch.x, train_batch.edge_index, train_batch.batch, train_batch.num_graphs)
    loss = F.cross_entropy(logits, train_batch.labels)
    loss.backward()
    if epoch == 0:
        checked = [model.layers[0].eps.grad, model.layers[0].mlp.fc1.weight.grad,
                   model.classifier.weight.grad]
        assert all(g is not None and torch.isfinite(g).all() and float(g.abs().sum()) > 0 for g in checked)
    optimizer.step()
    if epoch % 5 == 0:
        model.eval()
        with torch.no_grad():
            val_logits = model(val_batch.x, val_batch.edge_index, val_batch.batch, val_batch.num_graphs)
            val_acc = float((val_logits.argmax(1) == val_batch.labels).float().mean())
        if val_acc > best_val:
            best_val = val_acc
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}

assert best_state is not None
model.load_state_dict(best_state); model.eval()
with torch.no_grad():
    final_train_loss = float(F.cross_entropy(
        model(train_batch.x, train_batch.edge_index, train_batch.batch, train_batch.num_graphs),
        train_batch.labels))
    test_logits = model(test_batch.x, test_batch.edge_index, test_batch.batch, test_batch.num_graphs)
    test_acc = float((test_logits.argmax(1) == test_batch.labels).float().mean())
train_seconds = time.perf_counter() - train_start
assert final_train_loss < initial_loss * 0.35
assert best_val >= 0.99 and test_acc >= 0.99
assert train_seconds < 12.0
print({"initial_loss": round(initial_loss, 4), "final_loss": round(final_train_loss, 4),
       "val_acc": best_val, "test_acc": test_acc, "seconds": round(train_seconds, 3)})

## 7. 常见失败模式与生产边界

- **把 sum 改成 mean**：会削弱对多重集合计数的区分；是否使用 mean 应由任务假设决定，不能悄悄替换。
- **重复加入两个方向**：若原始无向边已展开，再次展开会把消息加倍。批处理函数只接受去重的无向 pair。
- **BatchNorm 污染**：图分开/合并执行时，BatchNorm 的统计会改变。本实现用逐节点 LayerNorm，使批隔离 oracle 更直接。
- **过平滑与过拟合**：层数增加并不总是更好；生产上需按图规模监控表征方差、图大小分桶指标和 OOD 拒识。
- **超大图**：Python 边循环只适合教学。生产应换成经过验证的稀疏 kernel，但必须保留方向、去重和跨图隔离测试。

权限过滤应在建图之前完成。图快照、特征模式、代码版本和权重必须共同绑定；仅保存一个 `.pt` 文件不足以复现实验。

In [ ]:
def state_hash(module: nn.Module) -> str:
    digest = hashlib.sha256()
    for name, tensor in sorted(module.state_dict().items()):
        digest.update(name.encode("utf-8")); digest.update(tensor.detach().cpu().contiguous().numpy().tobytes())
    return digest.hexdigest()[:20]

graph_snapshot = [{"id": g.graph_id, "n": int(g.x.shape[0]), "edges": list(g.edge_pairs),
                   "label": g.label, "split": g.split} for g in graphs]
artifact = {
    "architecture": "GINClassifier-3layer-sum-readout-v1",
    "seed": SEED,
    "feature_schema_hash": canonical_hash({"columns": ["constant", "local_parity"], "dtype": "float32"}),
    "graph_snapshot_hash": canonical_hash(graph_snapshot),
    "split_hash": canonical_hash({s: [g.graph_id for g in split_graphs[s]] for s in split_graphs}),
    "state_hash": state_hash(model),
    "metrics": {"best_val_accuracy": best_val, "test_accuracy": test_acc},
}
artifact["artifact_id"] = canonical_hash(artifact)

@dataclass(frozen=True)
class InferenceContext:
    scopes: frozenset[str]
    expected_artifact_id: str

def predict_graph(graph: GraphSample, ctx: InferenceContext) -> int:
    if "graph:predict" not in ctx.scopes:
        raise PermissionError("缺少 graph:predict")
    if ctx.expected_artifact_id != artifact["artifact_id"]:
        raise RuntimeError("请求绑定的 artifact 与当前服务不一致")
    item = batch_graphs([graph])
    with torch.no_grad():
        return int(model(item.x, item.edge_index, item.batch, 1).argmax(1).item())

ok_ctx = InferenceContext(frozenset({"graph:predict"}), artifact["artifact_id"])
assert predict_graph(graphs[-1], ok_ctx) in (0, 1)
assert len({artifact["feature_schema_hash"], artifact["graph_snapshot_hash"],
            artifact["split_hash"], artifact["state_hash"]}) == 4
try:
    predict_graph(graphs[-1], InferenceContext(frozenset(), artifact["artifact_id"]))
    raise AssertionError("缺少权限仍可推理")
except PermissionError as exc:
    assert "graph:predict" in str(exc)
print({"artifact_id": artifact["artifact_id"], "state_hash": artifact["state_hash"]})

## 8. 面试复盘与论文来源

一个完整回答应覆盖：为什么 GIN 使用 sum、`ε` 的作用、消息方向、多图块对角 batch、图级读出、置换不变性、切分协议，以及如何把权重绑定到图快照与特征模式。只写 `class GIN(nn.Module)` 而没有 oracle，无法证明实现的其实是 GIN。

主要来源：Xu, Hu, Leskovec & Jegelka, [**How Powerful are Graph Neural Networks?**](https://arxiv.org/abs/1810.00826), ICLR 2019；Morris et al., [**Weisfeiler and Leman Go Neural**](https://arxiv.org/abs/1810.02244), AAAI 2019。前者将 GIN 与 Weisfeiler–Lehman 图同构测试的表达力联系起来；本 Notebook 是小规模工程复现，不声称覆盖论文的全部理论与实验。